# Gold de geração distribuída — V-Energy

Consolida a geração distribuída (GD) integrando a Silver de GD com o cadastro de Unidades Consumidoras e o histórico mensal de leituras. Produz:

1. `gd_uc`: tabela de contrato do portal (`id_uc`, `potencia_kw`, `data_conexao`).
2. `tb_gd_uc`: dimensão enriquecida por UC com GD (`gd_uc_sk`, recortes cadastrais, consumo médio e mediano antes/após a conexão, queda percentual e `flag_queda_acima_30_pct`).
3. `tb_gd_consumo_mensal`: fato no grão `UC × competência` com `flag_gd_ativa_competencia` (`competencia >= competencia_conexao`), `volume_analise_kwh` e `consumo_kwh_analitico`.
4. `tb_gd_resumo_mensal`: agregado mensal por `competência × município × bairro × classe × modalidade`.

Regras e dicionário de dados: [`docs/gold/geracao_distribuida.md`](../../docs/gold/geracao_distribuida.md).

Por padrão, `modo=validar` executa todas as transformações, validações e exibições analíticas sem gravar no catálogo. Para publicar na camada Gold, ajuste `modo=publicar` (e opcionalmente informe `diretorio_exportacao` num Volume para exportar CSVs + `manifest.json`).

In [ ]:
# ruff: noqa: F821
%run ./gold_geracao_distribuida

In [ ]:
# ruff: noqa: F821
import re
from datetime import datetime, timezone
from pathlib import Path
from uuid import uuid4

from pyspark.sql import functions as F

parametros = {
    'tabela_gd': 'v-energy.silver.tb_geracao_distribuida',
    'tabela_cadastro': 'v-energy.silver.tb_unidades_consumidoras',
    'tabela_leituras': 'v-energy.silver.tb_leituras',
    'schema_gold': 'v-energy.gold',
    'diretorio_exportacao': '',
    'modo': 'validar',
}
for nome, padrao in parametros.items():
    dbutils.widgets.text(nome, padrao)
config = {nome: dbutils.widgets.get(nome).strip() for nome in parametros}
if config['modo'] not in {'validar', 'publicar'}:
    raise ValueError('modo deve ser validar ou publicar')


def qualificar(nome, partes=3):
    segmentos = nome.split('.')
    if len(segmentos) != partes or any(
        not re.fullmatch(r'[A-Za-z0-9_-]+', segmento) for segmento in segmentos
    ):
        raise ValueError(f'Identificador inválido: {nome}')
    return '.'.join(f'`{segmento}`' for segmento in segmentos)


schema_gold = qualificar(config['schema_gold'], 2)
if config['schema_gold'].split('.')[1] in {'silver', 'bronze', 'default'}:
    raise ValueError('O destino deve ser um schema próprio da Gold')
if config['modo'] == 'publicar' and config['diretorio_exportacao']:
    caminho = Path(config['diretorio_exportacao'])
    if not str(caminho).startswith('/Volumes/') or '..' in caminho.parts:
        raise ValueError(
            'Exportação exige diretorio_exportacao em /Volumes/catalogo/schema/volume/...'
        )
batch_id = uuid4().hex
fontes = {}
print(f'Schema Gold: {schema_gold} | Modo: {config["modo"]}')

## 1. Leitura consistente da Silver (`VERSION AS OF`)

Fixa a versão Delta atual de cada uma das três tabelas Silver no início da execução (`tabela_gd`, `tabela_cadastro` e `tabela_leituras`), garantindo que validações, transformações e escrita leiam exatamente o mesmo snapshot. Se os nomes físicos do cadastro ou das leituras estiverem em outro catálogo/schema no workspace (ex.: `workspace.default.silver_unidades_consumidoras` ou `venergy.silver.leituras`), informe-os nos widgets acima.

In [ ]:
# ruff: noqa: F821
def ler_snapshot(chave):
    tabela = qualificar(config[chave])
    if not spark.catalog.tableExists(tabela):
        raise ValueError(
            f'Tabela Silver não encontrada: {tabela}. '
            f'Publique a Silver correspondente ou ajuste o widget {chave}.'
        )
    detalhe = spark.sql(f'DESCRIBE DETAIL {tabela}').first()
    if detalhe['format'] != 'delta':
        raise ValueError(f'A fonte precisa ser Delta: {tabela}')
    historico = spark.sql(f'DESCRIBE HISTORY {tabela} LIMIT 1').first()
    fontes[chave] = {
        'tabela': config[chave],
        'id_delta': detalhe['id'],
        'versao_delta': historico['version'],
        'timestamp_commit': str(historico['timestamp']),
    }
    print(f'{chave}: {tabela} (versão {historico["version"]})')
    return spark.read.option('versionAsOf', historico['version']).table(tabela)


df_gd_silver = ler_snapshot('tabela_gd')
df_cadastro_silver = ler_snapshot('tabela_cadastro')
df_leituras_silver = ler_snapshot('tabela_leituras')

tabelas = construir_gold_gd(df_gd_silver, df_cadastro_silver, df_leituras_silver)
validar_reconciliacao(df_gd_silver, tabelas)
for nome, df_tab in tabelas.items():
    print(f'[OK] {nome}: {df_tab.count():,} registros validados em memória')

## 2. Verificação das perguntas de negócio do case

Confere diretamente nas tabelas Gold geradas:
- **Pergunta da Diretora de Tecnologia (Patrícia Lins):** quantas UCs com painel solar em Olinda tiveram queda de consumo acima de 30% após a conexão?
- **Pergunta do Diretor Comercial (Henrique Duarte):** qual foi a variação de volume das UCs com GD entre o primeiro ano (`P0`: jul/2022–jun/2023) e os últimos 12 meses (`P1`: jul/2025–jun/2026) por classe e município?
- **Evolução mensal:** estoque ativo de kWp e consumo antes × após a conexão.

In [ ]:
# ruff: noqa: F821
tb_gd_uc = tabelas['tb_gd_uc']
tb_resumo = tabelas['tb_gd_resumo_mensal']

print('=== 1) UCs com GD em Olinda e queda > 30% após a conexão ===')
display(
    tb_gd_uc.filter(F.upper(F.col('municipio')) == F.lit('OLINDA'))
    .agg(
        F.count('*').alias('total_ucs_gd_olinda'),
        F.count(F.when(F.col('flag_queda_acima_30_pct').isNotNull(), 1)).alias(
            'ucs_com_base_antes_e_apos'
        ),
        F.count(F.when(F.col('flag_queda_acima_30_pct'), 1)).alias('ucs_queda_acima_30_pct'),
    )
)

print('=== 2) Impacto de volume P0 (jul/22-jun/23) vs P1 (jul/25-jun/26) por classe ===')
display(
    tb_gd_uc.groupBy('classe')
    .agg(
        F.count('*').alias('qtd_ucs_gd'),
        F.sum('potencia_kwp').alias('potencia_total_kwp'),
        F.sum('volume_p0_utilizavel_kwh').alias('volume_p0_utilizavel_kwh'),
        F.sum('volume_p1_utilizavel_kwh').alias('volume_p1_utilizavel_kwh'),
        F.round(
            (F.sum('volume_p1_utilizavel_kwh') - F.sum('volume_p0_utilizavel_kwh'))
            * 100
            / F.sum('volume_p0_utilizavel_kwh'),
            2,
        ).alias('variacao_utilizavel_pct'),
        F.sum('volume_p0_analitico_kwh').alias('volume_p0_analitico_kwh'),
        F.sum('volume_p1_analitico_kwh').alias('volume_p1_analitico_kwh'),
        F.round(
            (F.sum('volume_p1_analitico_kwh') - F.sum('volume_p0_analitico_kwh'))
            * 100
            / F.sum('volume_p0_analitico_kwh'),
            2,
        ).alias('variacao_analitico_pct'),
    )
    .orderBy('classe')
)

print('=== 3) Evolução mensal do estoque ativo de GD ===')
display(
    tb_resumo.groupBy('competencia')
    .agg(
        F.sum('quantidade_novas_conexoes_mes').alias('novas_conexoes'),
        F.sum('quantidade_ucs_gd_ativas').alias('ucs_gd_ativas'),
        F.sum('potencia_instalada_ativa_kwp').alias('potencia_ativa_kwp'),
        F.sum('volume_analitico_kwh').alias('volume_analitico_total_kwh'),
    )
    .orderBy('competencia')
)

## 3. Publicação Delta com `RESTORE` e exportação CSV opcional

A publicação só ocorre quando `modo=publicar`. Antes da primeira escrita, registra a versão atual de cada tabela de destino e, se a escrita ou a conferência pós-escrita falhar, restaura os destinos afetados com `RESTORE TABLE ... TO VERSION AS OF` (ou `DROP TABLE IF EXISTS` na primeira carga).

In [ ]:
# ruff: noqa: F821
if config['modo'] == 'validar':
    print('Validação concluída. Nenhum schema, tabela ou CSV foi gravado.')
else:
    if any(dbutils.widgets.get(nome).strip() != valor for nome, valor in config.items()):
        raise ValueError(
            'Parâmetros mudaram durante a execução. Execute novamente desde o início.'
        )
    destinos = {nome: f'{schema_gold}.`{nome}`' for nome in tabelas}
    for chave, fonte in fontes.items():
        atual = spark.sql(f'DESCRIBE DETAIL {qualificar(config[chave])}').first()
        if atual['id'] != fonte['id_delta']:
            raise ValueError(f'A identidade da tabela fonte mudou: {chave}')
    if set(destinos.values()) & {qualificar(config[c]) for c in fontes}:
        raise ValueError('Uma tabela de destino coincide com uma fonte')

    spark.sql(f'CREATE SCHEMA IF NOT EXISTS {schema_gold}')
    persistidas, versoes_gold = publicar_delta(
        spark,
        tabelas,
        destinos,
        lambda gravadas: validar_reconciliacao(df_gd_silver, gravadas),
    )
    for nome, info in versoes_gold.items():
        print(f'Publicada: {info["tabela"]} (versão {info["versao_delta"]})')

    if config['diretorio_exportacao']:
        pasta_lote = exportar_csv(
            persistidas,
            str(Path(config['diretorio_exportacao']) / batch_id),
            {
                'batch_id': batch_id,
                'synthetic': False,
                'gerado_em_utc': datetime.now(timezone.utc).isoformat(),
                'fontes': fontes,
                'versoes_gold': versoes_gold,
            },
        )
        print(f'CSVs e manifesto exportados em: {pasta_lote}')